<h1><center>Laboratorio 5: La desperación de Mr. Lepin 🐼</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes


### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1:Jiale Chen
- Nombre de alumno 2:


---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores. 
- **Importante**: **¡Recuerden fijar semillas!** Así podemos reproducir sus resultados.

## Descripción del laboratorio.

### Importamos librerias utiles 😸

In [30]:
!uv add numpy pandas scikit-learn umap-learn plotly

Resolved 126 packages in 1ms
Checked 123 packages in 19ms


In [31]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, TransformerMixin


def plot_dim_reductions(
    pca_proj: np.ndarray,
    tsne_proj: np.ndarray,
    umap_proj: np.ndarray,
    name: None | str = None,
    colors: None | np.ndarray = None,
) -> go.Figure:
    fig = make_subplots(rows=1, cols=3, subplot_titles=("PCA", "t-SNE", "UMAP"))

    for i, (proj, title) in enumerate(zip([pca_proj, tsne_proj, umap_proj], ["PCA", "t-SNE", "UMAP"], strict=True)):
        temp_fig = px.scatter(
            x=proj[:, 0],
            y=proj[:, 1],
            color=colors.astype(str) if colors is not None else None,
            title=title,
            # showlegend=(i == 0),
        )

        for trace in temp_fig.data:
            trace.showlegend = i == 0
            fig.add_trace(trace, row=1, col=i + 1)

    fig.update_layout(height=400, width=1200, title_text=name)
    return fig

# Segmentación de Clientes en Tienda de Retail 🛍️

<p align="center">
  <img width=300 src="https://s1.eestatic.com/2018/04/14/social/la_jungla_-_social_299733421_73842361_854x640.jpg">
</p>

## 1.1 Cargar Dataset

Mr. Lepin, en una nueva reunión, le cuenta a ud y su equipo que los resultados derivados del análisis exploratorio de datos presentaron una gran utilidad para la empresa y que tiene un gran entusiasmo por continuar trabajando con ustedes.
Es por esto, que Mr. Lepin les pide que cargue y visualicen algunas de las filas que componen el Dataset.
A continuación un extracto de lo parlamentado en la reunión:

    - Usted: Es un gran logro para nuestro equipo que usted haya encontrado excelente el EDA. ¿Qué tiene en mente ahora?
    - Mr. Lepin: Resulta que hace algún tiempo, mientras tomaba un mojito en una reunión de gerentes en Panamá, oí a un *chato* acerca de **LRMFP**, que es un modelo que permite personificar a los clientes a través de la fabricación de distintos atributos que describen a los clientes. Lo encontré es-tu-pendo ñatito. 
    - Usted: Ehh bueno. Investigaremos acerca de este modelo y veremos lo que podemos hacer.

Por ende, su siguiente tarea es calcular **LRMFP** sobre cada cliente y luego hacer un análisis de las características generadas. Para esto, el área de ventas les entrega un nuevo archivo llamado `retail_dataset.pickle`, quien posee los datos del DataFrame original limpios y listos para obtener las características solicitadas por Mr. Lepin.

In [32]:
df_retail = pd.read_pickle(
    "https://github.com/MDS7202/MDS7202/raw/refs/heads/main/recursos/2026-01/labs/lab6/retail_dataset.pickle"
)
df_retail = df_retail.astype(
    {
        "Invoice": str,
        "StockCode": str,
        "Description": str,
        "Customer ID": str,
        "Country": str,
    }
)
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [33]:
df_retail[df_retail["Price"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country


## 1.2 Creación de nuevas Caracteristicas [2 Puntos] 

Como ya se les comentó, Mr. Lepin está interesado en obtener las características **LRMFP**, para esto les señala que estas características se construyen en base a las siguientes definiciones:

- **Length (L)**: Intervalo de tiempo, en días, entre la primera y la última visita del cliente. Mientras más grande sea el valor, más fiel es el cliente.

- **Recency (R)**: Indica hace cuánto tiempo el cliente realizó su última compra. Notar que para este caso, mientras más grande es el valor, menos interés posee el usuario para repetir una compra en uno de los locales.

- **Monetary (M)**: El término “monetario” se refiere a la cantidad media de dinero gastada por cada visita del cliente durante el período de observación y refleja la contribución del cliente a los ingresos de la empresa.

- **Frequency (F)**: Se refiere al número total de visitas del cliente durante el periodo de observación. Cuanto mayor sea la frecuencia, mayor será la fidelidad del cliente. 

- **Periodicity (P)**: Representa si los clientes visitan las tiendas con regularidad.

$$Periodicity(n)=std(IVT_1, ..., IVT_n)$$

&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Donde $IVT$ denota el tiempo entre visitas y n representa el número de valores de tiempo entre visitas de un cliente.
 

$$IVT_i=date\_diff(t_{i+1},t_i)$$

En base a las definiciones señaladas, diseñe una función que permita obtener las características **LRMFP** recibiendo un DataFrame como entrada. Para esto, no estará permitido el uso de iteradores; utilicen todas las herramientas que les ofrece `pandas` para realizar esto.

Una referencia que les puede ser útil es el [documento original](https://www.researchgate.net/publication/315979555_LRFMP_model_for_customer_segmentation_in_the_grocery_retail_industry_a_case_study) en donde se propone este método.

**<u>Formato</u> del Resultado Esperado:**

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

In [34]:
df_retail[df_retail["Customer ID"] == "12346.0"].sort_values("InvoiceDate")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
20724,491725,TEST001,This is a test product.,10,2009-12-14 08:34:00,4.50,12346.0,United Kingdom
20967,491742,TEST001,This is a test product.,5,2009-12-14 11:00:00,4.50,12346.0,United Kingdom
20968,491744,TEST001,This is a test product.,5,2009-12-14 11:02:00,4.50,12346.0,United Kingdom
27607,492718,TEST001,This is a test product.,5,2009-12-18 10:47:00,4.50,12346.0,United Kingdom
27618,492722,TEST002,This is a test product.,1,2009-12-18 10:55:00,1.00,12346.0,United Kingdom
30279,493410,TEST001,This is a test product.,5,2010-01-04 09:24:00,4.50,12346.0,United Kingdom
30280,493412,TEST001,This is a test product.,5,2010-01-04 09:53:00,4.50,12346.0,United Kingdom
36691,494450,TEST001,This is a test product.,5,2010-01-14 13:50:00,4.50,12346.0,United Kingdom
43488,495295,TEST001,This is a test product.,5,2010-01-22 13:30:00,4.50,12346.0,United Kingdom
76173,499763,15056P,EDWARDIAN PARASOL PINK,1,2010-03-02 13:08:00,5.95,12346.0,United Kingdom


**Respuesta:**

In [35]:
max_date = df_retail["InvoiceDate"].max()
max_date

Timestamp('2010-12-09 20:01:00')

In [36]:
def custom_features(dataframe_in: pd.DataFrame) -> pd.DataFrame:
    df = dataframe_in.copy()

    # Asegurar formato datetime
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

    # Crear columna de gasto por fila
    df["TotalPrice"] = df["Quantity"] * df["Price"]

    # Fecha máxima del dataset
    max_date = df["InvoiceDate"].max()

    # Agrupación por cliente
    grouped = df.groupby("Customer ID")

    # LENGTH: diferencia entre primera y última compra (se truncan la cantidad de días)
    length = (grouped["InvoiceDate"].max() - grouped["InvoiceDate"].min()).dt.days

    # RECENCY: tiempo desde última compra hasta fecha máxima del dataset
    # (se truncan la cantidad de días)
    recency = (max_date - grouped["InvoiceDate"].max()).dt.days

    # FREQUENCY: número de facturas únicas
    frequency = grouped["Invoice"].nunique()

    # MONETARY: gasto promedio por visita
    monetary = grouped["TotalPrice"].sum() / frequency

    # PERIODICITY:
    # Ordenar por cliente y fecha
    df_sorted = df.sort_values(["Customer ID", "InvoiceDate"])

    # Calcular diferencias entre visitas consecutivas
    df_sorted["IVT"] = df_sorted.groupby("Customer ID")["InvoiceDate"].diff().dt.days

    # Desviación estándar de IVT
    periodicity = df_sorted.groupby("Customer ID")["IVT"].std().fillna(0)
    # Construir DataFrame final
    result = pd.DataFrame(
        {"Length": length, "Recency": recency, "Frequency": frequency, "Monetary": monetary, "Periodicity": periodicity}
    ).reset_index()

    return result

In [37]:
custom_features(df_retail)

,Customer ID,Length,Recency,Frequency,Monetary,Periodicity
0,12346.0,196,164,11,33.896364,21.724076
1,12347.0,37,2,2,661.660000,4.422346
2,12348.0,0,73,1,222.160000,0.000000
3,12349.0,181,42,3,890.380000,16.200990
4,12351.0,0,10,1,300.930000,0.000000
...,...,...,...,...,...,...
4309,18283.0,275,17,6,103.228333,11.783701
4310,18284.0,0,66,1,461.680000,0.000000
4311,18285.0,0,295,1,427.000000,0.000000
4312,18286.0,247,111,2,648.215000,30.403598


## 1.3 Pipelines 👷

Finalmente *Mr. Lepin* le pregunta si sería posible realizar un pipeline para realizar una segmentación de los clientes con los nuevos datos generados, a lo que usted responde que **sí** y propone la utilización de k-means para la segmentación.

A continuación siga los pasos requeridos para obtener la segmentación de clientes.

### 1.3.1 Estandarizar Caracteristicas [0.5 puntos]

Construya una clase llamada ``MinMax()`` utilizando ``BaseEstimator`` y ``TransformerMixin`` para realizar una transformación de cada una de las columnas de un DataFrame utilizando ``ColumnTransformer()`` más tarde (tome como referencia el siguiente [enlace](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer)).


 Para esto considere que Min-Max escaler queda dada por la ecuación:

$$MinMax = \dfrac{x-min(x)}{max(x) - min(x)}$$

Con esto buscamos que los valores que componen a las columnas se muevan en el rango de valores $[0, 1]$.

**Respuesta:**

In [38]:
class MinMax(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = np.array(X)
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self

    def transform(self, X):
        X = np.array(X)
        diff = self.max_ - self.min_
        return np.where(diff == 0, 0, (X - self.min_) / diff)

    # si no hay diferencia en el mínimo y máximo, se retorna 0

### 1.3.2 Pipelines de Proyecciones [0.5 puntos]

Para comparar técnicas de reducción de dimensionalidad, realice **tres pipelines** distintos sobre los datos **LRMFP** usando los siguientes métodos:
- **PCA**
- **t-SNE**
- **UMAP**

Para cada pipeline, siga estos pasos:
1. Obtenga las características **LRMFP** desde el DataFrame `retail_dataset.pickle` utilizando la función ``custom_features`` creada anteriormente, junto a ``FunctionTransformer()``. Considere esto como el primer paso de su pipeline.
2. En segundo lugar, usando ``ColumnTransformer()``, aplique el MinMax scaler creado por usted sobre todas las columnas generadas en el paso anterior.
3. Finalmente, aplique el método de reducción de dimensionalidad correspondiente (PCA, t-SNE o UMAP) para obtener las 2 componentes más relevantes.

A continuación, grafique las proyecciones obtenidas de las tres técnicas en una sola figura comparativa.

**Respuesta:**

In [39]:
import umap.umap_ as umap
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

random_seed = 123

features_transformer = FunctionTransformer(custom_features)

scaler = ColumnTransformer(
    transformers=[("minmax", MinMax(), ["Length", "Recency", "Frequency", "Monetary", "Periodicity"])]
)

pipeline_pca = Pipeline(
    [("features", features_transformer), ("scaling", scaler), ("pca", PCA(n_components=2, random_state=random_seed))]
)
pipeline_tsne = pipeline_tsne = Pipeline(
    [("features", features_transformer), ("scaling", scaler), ("tsne", TSNE(n_components=2, random_state=random_seed))]
)
pipeline_umap = Pipeline(
    [
        ("features", features_transformer),
        ("scaling", scaler),
        ("umap", umap.UMAP(n_components=2, random_state=random_seed)),
    ]
)

In [40]:
# Utilice este código para ejecutar las pipelines y graficar.

pca_proj = pipeline_pca.fit_transform(df_retail)
tsne_proj = pipeline_tsne.fit_transform(df_retail)
umap_proj = pipeline_umap.fit_transform(df_retail)

fig = plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="Reducción de Dimensionalidad", colors=None)
fig.show()

c:\MDS7202\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### 1.3.3 Análisis de los Loadings de PCA [0.5 puntos]
Antes de continuar con la etapa de clustering, analice los *loadings* (pesos o coeficientes) de las componentes principales obtenidas con PCA. 

Utilice el siguiente tutorial para visualizarlos: https://plotly.com/python/pca-visualization/

- Calcule y reporte los *loadings* de las dos primeras componentes principales.
- Interprete qué características (**LRMFP**) son más relevantes en cada componente.
- Visualice los *loadings* usando un gráfico de barras para cada componente.



In [41]:
# Código para calcular loadings.

# Obtener el PCA del pipeline ya fiteado
pca_model = pipeline_pca.named_steps["pca"]

# Loadings = eigenvectors * sqrt(eigenvalues)
loadings = pca_model.components_.T * np.sqrt(pca_model.explained_variance_)

loadings_df = pd.DataFrame(
    loadings, index=["Length", "Recency", "Frequency", "Monetary", "Periodicity"], columns=["PC1", "PC2"]
)

print("Loadings de las dos primeras componentes principales:")
print(loadings_df.round(4))

Loadings de las dos primeras componentes principales:
                PC1     PC2
Length       0.3428  0.0960
Recency     -0.1863  0.1812
Frequency    0.0179  0.0040
Monetary     0.0018 -0.0005
Periodicity  0.0458  0.0173


### Interpretación de las características según los loadings

**PC1:**  
La primera componente principal está influenciada principalmente por **Length** (positiva) y **Recency** (negativa). Esto indica que captura la relación temporal activa del cliente, diferenciando entre clientes con larga trayectoria y compras recientes frente a aquellos menos activos o con menor historial.

**PC2:**  
La segunda componente principal está dominada por **Recency** (positiva), con una menor contribución de **Length**. Representa principalmente el nivel de inactividad del cliente, donde valores altos indican clientes que llevan más tiempo sin comprar.

In [42]:
loadings_long = loadings_df.reset_index().melt(
    id_vars="index", value_vars=["PC1", "PC2"], var_name="Component", value_name="Loading"
)

loadings_long.rename(columns={"index": "Feature"}, inplace=True)

fig = px.bar(
    loadings_long, x="Feature", y="Loading", color="Component", barmode="group", title="Loadings PCA: PC1 vs PC2"
)

fig.show()

In [43]:
features = loadings_df.index

fig = px.scatter(
    x=pca_proj[:, 0], y=pca_proj[:, 1], title="PCA Loadings Visualization (LRMFP)", labels={"x": "PC1", "y": "PC2"}
)

# Escala para que las flechas sean visibles
scale = 3

for i, feature in enumerate(features):
    x_loading = loadings_df.iloc[i, 0] * scale
    y_loading = loadings_df.iloc[i, 1] * scale

    # Flecha (vector desde el origen)
    fig.add_annotation(
        ax=0,
        ay=0,
        axref="x",
        ayref="y",
        x=x_loading,
        y=y_loading,
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top",
    )

    # Etiqueta
    fig.add_annotation(
        x=x_loading,
        y=y_loading,
        ax=0,
        ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )

fig.update_layout(width=1200, height=600)
fig.show()

### Preguntas sobre loadings:

- ¿Qué son los loadings de PCA?

> Respuesta: Los *loadings* en PCA son los coeficientes que indican cuánto contribuye cada variable original a cada componente principal, representando su peso en la construcción de las nuevas dimensiones. Se derivan de los vectores propios de la matriz de covarianza y permiten interpretar las componentes principales, ya que valores altos (positivos o negativos) indican una fuerte influencia de la variable, mientras que valores cercanos a cero reflejan una contribución baja.

- ¿Qué información relevante obtiene sobre la estructura de los datos a partir de los *loadings* de PCA?

> Respuesta: Los *loadings* de PCA muestran que la estructura del dataset está dominada por variables temporales, específicamente **Length** y **Recency**, que concentran la mayor variabilidad. En PC1 ambas variables presentan una relación inversa, indicando un eje que contrapone antigüedad y actividad reciente, mientras que en PC2 capturan principalmente el nivel de inactividad. En contraste, **Frequency** y **Monetary** tienen una contribución prácticamente nula, lo que indica que no son relevantes para explicar la estructura principal de los datos. En consecuencia, la información más relevante del dataset está asociada a patrones temporales de comportamiento más que a la intensidad de compra.

- ¿Existe alguna relación interesante entre las direcciones de las variables?

> Respuesta: Sí, se observa una relación clara entre las direcciones de las variables. En particular, **Length** y **Recency** presentan direcciones opuestas en la primera componente principal, lo que indica una relación inversa entre antigüedad y recencia. Además, **Recency** tiene una componente importante en el eje vertical (PC2), lo que sugiere que también captura variabilidad adicional relacionada con la inactividad del cliente. En contraste, **Frequency**, **Monetary** y **Periodicity** se concentran cerca del origen, indicando una baja influencia y ausencia de relaciones estructurales relevantes en las principales componentes.

## 1.4 Clustering

### 1.4.1 Método del Codo [0.5 puntos]

Utilizando la clase creada para escalamiento, aplique el método del codo para visualizar cuál es el número de clusters que mejor se ajustan a los datos. Realice esto utilizando el algoritmo K-means dentro de un pipeline para un $k \in [1,20]$, donde k representa el número de clusters del k-means. Para la realización de esta sección y la próxima (1.4.2), considere los mismos pasos utilizados para el t-SNE, pero **permutando el algoritmo de reducción de dimensionalidad por k-means.**

**Respuesta:**

In [44]:
from sklearn.cluster import KMeans  # noqa: F401

k_values = range(1, 21)
inertias = []

for k in k_values:
    pipeline_kmeans = Pipeline(
        [
            ("features", FunctionTransformer(custom_features)),
            (
                "scaling",
                ColumnTransformer(
                    transformers=[("minmax", MinMax(), ["Length", "Recency", "Frequency", "Monetary", "Periodicity"])]
                ),
            ),
            ("kmeans", KMeans(n_clusters=k, random_state=random_seed, n_init="auto")),
        ]
    )

    pipeline_kmeans.fit(df_retail)
    inertias.append(pipeline_kmeans.named_steps["kmeans"].inertia_)

In [45]:
fig = px.line(
    x=list(k_values),
    y=inertias,
    markers=True,
    title="Método del Codo para K-Means",
    labels={"x": "Número de clusters (k)", "y": "Inercia"},
)

fig.update_layout(width=800, height=500)
fig.show()

### Preguntas Método del Codo

- A través del gráfico obtenido, comente y justifique qué valor de k escogería para realizar el k-means.

> Respuesta: Del gráfico se observa una disminución pronunciada de la inercia hasta aproximadamente \(k = 4\). A partir de ese punto, la caída entre \(k=4\) y \(k=5\) es casi igual a la de \(k=5\) a \(k=6\), lo que indica que las mejoras son marginales. Por ello, no resulta conveniente agregar más clusters y se elige \(k = 4\) como valor adecuado.

- Le fue útil el método del codo para encontrar el número de clusters?

> Respuesta: El método del codo no fue completamente útil en este caso (sirve de baseline), ya que el gráfico no presenta un punto de inflexión claramente definido que permita identificar de forma determinística el valor óptimo de \(k\). La curva muestra una disminución progresiva de la inercia sin un “codo” marcado, lo que deja margen a distintas interpretaciones al momento de seleccionar el número de clusters.

- Si no fue así, ¿qué otros métodos podría haber usado para encontrar un número óptimo de clusters?

> Respuesta: Se pueden utilizar métricas como el **coeficiente de Silhouette**, que mide la separación y cohesión de los clusters; el **índice de Calinski-Harabasz**, que evalúa la relación entre dispersión entre e intra clusters; y el **índice de Davies-Bouldin**, que cuantifica la similitud entre clusters (menor es mejor). También se puede usar el **Gap Statistic**, que compara la estructura observada con datos aleatorios para estimar un \(k\) óptimo.

### 1.4.2 Segmentación de Clientes con K-Means 🎁 [1 punto]

Por último, Mr. Lepin, impaciente de no entender lo que usted intenta explicarle, le solicita que por favor muestre algún resultado "visual y entendible" de los grupos encontrados.

En base a la elección de k realizada en la sección anterior, utilice este valor escogido y entrene un modelo de K-means utilizando el mismo pipeline de scikit-learn utilizado anteriormente.

Una vez ajustado los datos, genere una tabla con los promedios (o medianas) para cada uno de los atributos, agrupando estos por el clúster que pertenecen.

Finalmente, construya un heatmap de las características promedio de cada cluster para visualizar y comparar los perfiles de los grupos.

**Estadísticas de Referencia para K=6:**

Ud. debe calcularlas - Varían de ejecución en ejecución.


|         | Length  | Recency   | Frequency | Monetary | Periodicity |       |
|---------|---------|-----------|----------|-------------|-------|-------|
| Cluster |         |           |          |             |       |       |
|    0    |   258.8 |      45.2 |     76.1 |      1107.7 | 107.6 |   449 |
|    1    |    76.1 |     217.6 |     45.5 |       791.7 |  14.1 |   466 |
|    2    |   368.5 |       4.8 |   2715.0 |    226621.6 |   4.2 |     4 |
|    3    |    85.3 |      45.7 |     65.8 |      1047.0 |  10.5 |   987 |
|    4    |   347.2 |      15.9 |   1658.0 |     35829.3 |   8.0 |    25 |
|    5    |   298.0 |      29.8 |    183.8 |      3639.9 |  32.0 |  1188 |

In [56]:
# Aquí calcule K-Means
k_optimo = 4

pipeline_kmeans = Pipeline(
    [
        ("features", FunctionTransformer(custom_features)),
        (
            "scaling",
            ColumnTransformer(
                transformers=[("minmax", MinMax(), ["Length", "Recency", "Frequency", "Monetary", "Periodicity"])]
            ),
        ),
        ("kmeans", KMeans(n_clusters=k_optimo, random_state=random_seed, n_init="auto")),
    ]
)

pipeline_kmeans.fit(df_retail)

clusters = pipeline_kmeans.named_steps["kmeans"].labels_

df_features = custom_features(df_retail)
df_features["Cluster"] = clusters

cluster_mean = df_features.groupby("Cluster")[["Length", "Recency", "Frequency", "Monetary", "Periodicity"]].mean()

cluster_median = df_features.groupby("Cluster")[["Length", "Recency", "Frequency", "Monetary", "Periodicity"]].median()

display(cluster_mean)
display(cluster_median)

,Length,Recency,Frequency,Monetary,Periodicity
Cluster,,,,,
0,17.749179,254.167579,1.508215,351.139332,2.847132
1,181.929563,66.490079,4.198413,364.981625,20.846738
2,17.262334,50.881754,1.722005,372.157043,2.394149
3,319.146953,22.731183,10.221326,411.456732,20.131680


,Length,Recency,Frequency,Monetary,Periodicity
Cluster,,,,,
0,0.0,247.0,1.0,228.300000,0.000000
1,182.0,58.0,3.0,295.185000,17.270747
2,0.0,43.0,1.0,268.280000,0.000000
3,326.0,16.0,7.0,323.663808,14.147282


In [57]:
# Utilice la siguiente función para graficar k-means. kmeans_labels = clusters obtenidos por k-means.
kmeans_labels = clusters
plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="KMeans K=4", colors=kmeans_labels)

In [53]:
# Aquí grafique el Heatmap
fig = px.imshow(
    cluster_mean,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    title="Perfil promedio de cada cluster (LRMFP)",
    labels={"x": "Características", "y": "Cluster", "color": "Valor promedio"},
)

fig.update_layout(width=850, height=500)

fig.show()

### Preguntas sobre K-Means: 

- ¿Se separaron bien los distintos clusters en cada visualización? 

> Respuesta: La separación de los clusters resulta difícil de visualizar claramente en las tres proyecciones. En **PCA** se observa un solapamiento importante entre grupos debido a su naturaleza lineal, mientras que en **t-SNE** y **UMAP**, aunque los puntos tienden a agruparse, los clusters no están completamente separados y presentan cercanía entre sí, lo que dificulta una delimitación clara. Sin embargo, el **heatmap** evidencia que los clusters sí difieren en sus características promedio, especialmente en variables como **Recency**, **Length** y **Monetary**. Esto sugiere que, aunque la separación visual no es completamente clara en el espacio reducido, los grupos capturan diferencias reales en los perfiles de los clientes.

- ¿Es posible observar agrupaciones coherentes?

> Respuesta: Sí, es posible observar agrupaciones coherentes, aunque no estén claramente separadas en el espacio visual. En las proyecciones (PCA, t-SNE y UMAP) se identifican zonas donde los puntos tienden a agruparse, pero con cierto solapamiento entre clusters. Sin embargo, el heatmap muestra diferencias consistentes en las características promedio de cada grupo, lo que indica que los clusters representan perfiles distintos de clientes. En conjunto, esto sugiere que las agrupaciones son coherentes en términos de sus atributos, aunque su separación visual no sea completamente nítida.

- ¿Quedarían mejor más o menos clusters?

> Respuesta: No es evidente que aumentar o disminuir el número de clusters mejore significativamente la segmentación. Dado que incluso con \(k = 4\) existe solapamiento en las visualizaciones, aumentar \(k\) podría generar grupos más fragmentados pero no necesariamente más interpretables. Por otro lado, disminuir \(k\) podría simplificar la estructura, pero a costa de perder diferencias relevantes entre perfiles.

En consecuencia, el valor elegido (\(k = 4\)) representa un compromiso razonable entre simplicidad e interpretabilidad. Sin embargo, para una decisión más robusta, se podrían complementar estos resultados utilizando otros métodos de evaluación, como el coeficiente de Silhouette, el índice de Calinski-Harabasz o el índice de Davies-Bouldin, permitiendo comparar distintas configuraciones y seleccionar un \(k\) óptimo de forma más fundamentada.

- ¿K-Means, dada la forma de las proyecciones, será el mejor método para clusterizar este dataset?¿Habrá algún otro mejor?

> Respuesta: K-Means no parece ser el método más adecuado para este dataset, ya que, al ser particional, requiere fijar \(k\), asigna cada punto a un único cluster y asume grupos de forma esférica, lo cual no se ajusta bien a las proyecciones observadas, donde existe solapamiento y formas no claramente convexas. En este contexto, métodos como **DBSCAN** (basado en densidad) podrían ser más adecuados al no requerir \(k\), permitir clusters de forma arbitraria y detectar outliers; los métodos **jerárquicos** permiten explorar distintas particiones sin fijar previamente el número de clusters; y modelos **difusos** como **GMM** ofrecen mayor flexibilidad al asumir formas elipsoidales y permitir asignaciones probabilísticas. En conjunto, estos enfoques podrían capturar mejor la estructura del dataset que K-Means.

Y por último:

- Nombre a cada uno de los clusters según el comportamiento de sus miembros (ej. "C1: Compran poco pero con gran valor...") - Si es necesario, ajuste el número de clusters antes de responder.

> Respuestas: Los clusters pueden interpretarse y nombrarse de la siguiente manera:

A partir del heatmap y de las definiciones LRMFP, \(k = 4\) parece razonable, ya que permite distinguir perfiles con comportamientos diferentes. Los clusters pueden nombrarse así:

- **C0: Clientes inactivos y poco fieles**  
  Presentan bajo **Length**, baja **Frequency**, baja **Periodicity** y alta **Recency**, por lo que corresponden a clientes con poca historia, pocas visitas y que llevan bastante tiempo sin comprar.

- **C1: Clientes moderadamente fieles pero irregulares**  
  Tienen **Length** medio-alto, **Recency** moderada, **Frequency** intermedia y alta **Periodicity**, lo que indica clientes con cierta antigüedad y actividad, pero con visitas poco regulares.

- **C2: Clientes recientes de baja actividad**  
  Presentan bajo **Length**, baja **Frequency** y baja **Periodicity**, pero menor **Recency** que C0. Esto sugiere clientes con poca historia y pocas compras, aunque más recientes.

- **C3: Clientes fieles y de alto valor**  
  Tienen el mayor **Length**, mayor **Frequency**, mayor **Monetary** y baja **Recency**, por lo que corresponden a clientes fieles, frecuentes, activos y con mayor contribución monetaria.


Justifique su respuesta y no decepcione a Mr. Lepin.

## 1.5 Detección de Anomalías con DBSCAN [1 punto]
En esta sección, utilizará el algoritmo DBSCAN para identificar posibles anomalías (outliers) en los clientes del retail.

- Puede aplicar DBSCAN sobre las características originales escaladas (**LRMFP**) o sobre alguna de las proyecciones 2D (PCA, t-SNE o UMAP). Justifique su elección en las preguntas al final de la sección.
- Visualice los resultados usando `plot_dim_reductions`, mostrando los clusters y resaltando los outliers (label = -1) en las tres proyecciones (PCA, t-SNE, UMAP).

In [63]:
from sklearn.cluster import DBSCAN  # noqa: F401

pipeline_dbscan = Pipeline(
    [
        ("features", FunctionTransformer(custom_features)),
        (
            "scaling",
            ColumnTransformer(
                transformers=[("minmax", MinMax(), ["Length", "Recency", "Frequency", "Monetary", "Periodicity"])]
            ),
        ),
        ("dbscan", DBSCAN(eps=0.1, min_samples=10)),
    ]
)

dbscan_labels = pipeline_dbscan.fit_predict(df_retail)

In [64]:
# Utilice este código para graficar. dbscan_labels = clusters/outliers obtenidos por DBSCAN.
fig_dbscan = plot_dim_reductions(
    pca_proj,
    tsne_proj,
    umap_proj,
    name="DBSCAN - Detección de Anomalías",
    colors=dbscan_labels,
)
fig_dbscan.show()

In [73]:
df_features = custom_features(df_retail)
df_features["DBSCAN_Label"] = dbscan_labels

outliers_df = df_features[df_features["DBSCAN_Label"] == -1]

display(outliers_df.describe())
display(df_features.describe())

,Length,Recency,Frequency,Monetary,Periodicity,DBSCAN_Label
count,94.000000,94.000000,94.000000,94.000000,94.000000,94.0
mean,218.840426,75.617021,18.851064,1627.101326,56.337537,-1.0
std,121.205797,85.792837,39.701022,2475.009142,51.607661,0.0
min,0.000000,0.000000,1.000000,19.460000,0.000000,-1.0
25%,144.000000,10.000000,2.000000,133.912500,3.039791,-1.0
50%,235.000000,41.000000,2.000000,335.139903,58.383611,-1.0
75%,331.000000,98.000000,6.000000,2530.210139,91.925967,-1.0
max,373.000000,289.000000,205.000000,11880.840000,181.169120,-1.0


,Length,Recency,Frequency,Monetary,Periodicity,DBSCAN_Label
count,4314.000000,4314.000000,4314.000000,4314.000000,4314.000000,4314.000000
mean,133.936486,90.269124,4.454103,376.198874,11.390177,-0.021790
std,132.827714,96.943482,8.168658,492.309973,16.083429,0.146013
min,0.000000,0.000000,1.000000,0.000000,0.000000,-1.000000
25%,0.000000,17.000000,1.000000,179.400000,0.000000,0.000000
50%,105.000000,52.000000,2.000000,284.855000,6.974007,0.000000
75%,253.750000,135.000000,5.000000,421.450000,16.547550,0.000000
max,373.000000,373.000000,205.000000,11880.840000,181.169120,0.000000


In [62]:
configs = [(0.05, 5), (0.1, 5), (0.15, 10), (0.2, 15)]

for eps, min_samples in configs:
    pipeline_dbscan = Pipeline(
        [
            ("features", FunctionTransformer(custom_features)),
            (
                "scaling",
                ColumnTransformer(
                    transformers=[("minmax", MinMax(), ["Length", "Recency", "Frequency", "Monetary", "Periodicity"])]
                ),
            ),
            ("dbscan", DBSCAN(eps=eps, min_samples=min_samples)),
        ]
    )

    dbscan_labels = pipeline_dbscan.fit_predict(df_retail)

    fig_dbscan = plot_dim_reductions(
        pca_proj,
        tsne_proj,
        umap_proj,
        name=f"DBSCAN - eps={eps}, min_samples={min_samples}",
        colors=dbscan_labels,
    )

    fig_dbscan.show()

### Preguntas sobre DBSCAN


1. ¿Por qué decidiste usar los datos originales completos o las proyecciones para aplicar DBSCAN? ¿Por qué no usaste la otra opción?

> Respuesta: Se decidió aplicar DBSCAN sobre las características originales **LRMFP escaladas**, ya que estas preservan la estructura real del espacio de datos y las distancias entre observaciones, las cuales son fundamentales para un algoritmo basado en densidad como DBSCAN. En cambio, no se utilizaron las proyecciones (PCA, t-SNE o UMAP) porque estas transformaciones reducen la dimensionalidad y pueden distorsionar las distancias originales: PCA pierde información al comprimir dimensiones, mientras que t-SNE y UMAP priorizan la estructura local para visualización, alterando la densidad global. Por esta razón, aplicar DBSCAN sobre las proyecciones podría llevar a identificar clusters o outliers que no reflejan la verdadera estructura del dataset.

2. ¿Cómo elegiste los parámetros de DBSCAN (`eps`, `min_samples`)? ¿Probaste diferentes valores? ¿Cómo afectó esto los resultados?

> Respuesta: Los parámetros de DBSCAN se eligieron mediante una combinación de criterios empíricos y experimentación. Inicialmente, se utilizó \(eps = 0.1\) considerando que los datos fueron escalados en el rango \([0,1]\), y \(min\_samples = 10\) siguiendo la regla práctica de aproximadamente el doble del número de variables (5 en LRMFP). 

Posteriormente, se probaron distintas configuraciones (\(eps = 0.05, 0.1, 0.15, 0.2\) y \(min\_samples = 5, 10, 15\)) para evaluar su impacto. Se observó que con valores pequeños de \(eps\) (por ejemplo, 0.05) el modelo detecta muchos outliers y pocos clusters, ya que la condición de densidad es muy estricta. A medida que \(eps\) aumenta, disminuye la cantidad de outliers y los puntos tienden a agruparse en un único cluster dominante, perdiendo capacidad de detección de anomalías. Por otro lado, al aumentar \(min\_samples\), la definición de densidad se vuelve más exigente, lo que incrementa la cantidad de puntos clasificados como outliers.

En conjunto, los resultados muestran que DBSCAN es altamente sensible a estos parámetros, y que su ajuste implica un balance entre detectar estructuras densas y evitar clasificar excesivamente puntos como ruido.

3. ¿Tienen sentido los outliers encontrados según el contexto del negocio? ¿Qué interpretación le das a estos clientes? Analiza los datos con pandas si es necesario.

> Respuesta: Sí, los outliers encontrados tienen sentido dentro del contexto del negocio y representan clientes con comportamientos extremos respecto al resto.

Al comparar los outliers con los clientes normales, se observa que los outliers presentan valores significativamente más altos en **Monetary**, **Frequency** y especialmente **Periodicity**. Por ejemplo, el gasto promedio de los outliers (~1627) es muy superior al de los clientes normales (~376), y también presentan una mayor frecuencia y variabilidad en sus visitas. Esto indica que estos clientes tienen patrones de compra mucho más intensos o irregulares que el promedio.

Además, los outliers tienen en promedio menor **Recency** que los clientes normales, lo que sugiere que, en general, son más recientes o activos, aunque también presentan una alta dispersión en esta variable. La **Periodicity** elevada indica que sus compras no siguen un patrón regular, lo que refuerza su carácter atípico.

En términos de negocio, estos outliers pueden interpretarse como:
- **Clientes de alto valor**: alto gasto y frecuencia.
- **Clientes con comportamiento irregular**: compras grandes pero poco predecibles.
- **Casos especiales o extremos**: como compras excepcionales o clientes con hábitos fuera del patrón general (periodicidad variante).

En conclusión, los outliers no corresponden a ruido irrelevante, sino a clientes con perfiles extremos y potencialmente importantes, que podrían requerir estrategias diferenciadas de análisis.

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por correo, Discord o U-cursos.

![Gracias Totales!](https://i.pinimg.com/originals/65/ae/27/65ae270df87c3c4adcea997e48f60852.gif "bruno")


<br>
<center>
<img src="https://i.kym-cdn.com/photos/images/original/001/194/195/b18.png" width=100 height=50 />
</center>
<br>